In [1]:
import os
import pandas as pd
import numpy as np
import glob

from datetime import datetime, timedelta
from astropy.time import Time
import gc

# functions

In [1]:
def filter_intervals(triggers, gps_start, gps_end, intervals):
    
    intervals_crop = intervals[(intervals['int2'] >= gps_start) & (intervals['int1'] <= gps_end)].copy().sort_values('int1')
    intervals_crop['int1'] = intervals_crop['int1'].astype('float64')

    triggers_crop = triggers[(triggers['time'] >= gps_start) & (triggers['time'] <= gps_end)].copy().sort_values('time')

    if triggers_crop.empty or intervals_crop.empty:
        return pd.DataFrame(columns=triggers.columns)

    merged = pd.merge_asof(triggers_crop, intervals_crop, left_on='time', right_on='int1', direction='backward')
    # valid = merged[(merged['time'] >= merged['int1']) & (merged['time'] <= merged['int2'])]
    valid = merged[merged['time'] <= merged['int2']].copy()

    triggers_final = valid.drop(columns=['int1', 'int2'], errors='ignore')

    return triggers_final

In [2]:
def clusteringGlitches(gltab, min_triggers=20, deltat=0.2):
    
    if len(gltab) == 0:
        return pd.DataFrame()

    table = gltab.sort_values(by='time').reset_index(drop=True).copy()

    times = table['time'].values
    deltas = np.diff(times)

    new_cluster_mask = np.concatenate([[False], deltas > deltat])
    table['cluster'] = np.cumsum(new_cluster_mask)

    counts = table['cluster'].value_counts().sort_index()
    
    valid_clusters = counts[counts >= min_triggers].index
    table_filtered = table[table['cluster'].isin(valid_clusters)].copy()

    if table_filtered.empty:
        return pd.DataFrame()

    unique_clusters = table_filtered['cluster'].unique()
    cluster_mapping = {old_id: new_id for new_id, old_id in enumerate(unique_clusters)}
    table_filtered['cluster'] = table_filtered['cluster'].map(cluster_mapping)

    df_clustered = table_filtered.copy()
    
    if df_clustered.empty:
        return pd.DataFrame()

    summary_list = []

    for cl, group in df_clustered.groupby('cluster'):
        max_row = group.loc[group['snr'].idxmax()]

        summary_list.append({
            'cluster': cl,
            'num_triggers': len(group),
            'time': max_row['time'],             # GPS Time oficial (dt=0 no pico SNR)
            'frequency': max_row['frequency'],   # Frequência central no pico
            'snr': max_row['snr'],               # SNR Máximo
            'tstart': group['time'].min(),
            'tend': group['time'].max(),
            'min_freq': group['frequency'].min(),
            'max_freq': group['frequency'].max()
        })

    return pd.DataFrame(summary_list)

In [3]:
def routine(current_day, end_day, unclustered, intervals, min_triggers, deltat, output_dir, global_cluster_ID=0):

    os.makedirs(output_dir, exist_ok=True)
    trigger_count = []
    glitch_count = []

    all_glitches = []
    
    while current_day <= end_day:
        
        date_str = current_day.strftime('%Y-%m-%d')
        out_file = os.path.join(output_dir, f"glitches_{date_str}.parquet")

        if os.path.exists(out_file):
            print(f"Day {date_str} already processed. Skipping...")

            try:
                df_exist = pd.read_parquet(out_file)
                if not df_exist.empty:
                    
                    global_cluster_ID = df_exist['cluster'].max() + 1
                    
            except Exception:
                pass
            
            current_day += timedelta(days=1)
            continue

        day_start = datetime(current_day.year, current_day.month, current_day.day, 0, 0, 0)
        day_end   = datetime(current_day.year, current_day.month, current_day.day, 23, 59, 59)

        print(f"Processing day: {current_day.date()}")

        gps_start = Time(day_start, scale="utc").gps
        gps_end   = Time(day_end, scale="utc").gps

        # CONSIDER THE INTERVALS OF THE INTERFEROMETER ------------------------------
        gltab = unclustered[(unclustered['time'] >= gps_start) & (unclustered['time'] <= gps_end)].copy()
        
        gltab = filter_intervals(gltab, gps_start, gps_end, intervals)
        gltab = gltab.sort_values(by="time").reset_index(drop=True)
        
        # CLUSTERING OF THE CURRENT DAY
        if not gltab.empty:
            clustered = clusteringGlitches(gltab, min_triggers, deltat)

        if not clustered.empty:
            # CREATE GLITCHGRAMS FOR CURRENT DAY ------------------------------
            clustered['cluster'] += global_cluster_ID
            global_cluster_ID = clustered['cluster'].max() + 1
            
            abs_out_file = os.path.abspath(out_file)
    
            try:
                clustered.to_parquet(abs_out_file, index=False)
                print(f"Successfully saved: {out_file}\n")
            except Exception as e:
                csv_file = abs_out_file.replace('.parquet', '.csv')
                clustered.to_csv(csv_file, index=False)
                print(f"Parquet failed ({e}). SAVED IN CSV: {csv_file}\n")

            all_glitches.append(clustered.copy())
            del clustered
        else:
            print(f"No valid clusters found for day {date_str}.\n")
            gltab = gltab.sort_values(by="time").reset_index(drop=True)

        del gltab
        gc.collect()
        
        current_day += timedelta(days=1)
        
    print(f"Processing complete. Next available global_cluster_ID: {global_cluster_ID}")
    
    if all_glitches:
        return pd.concat(all_glitches, ignore_index=True)
    else:
        return pd.DataFrame()

# routine

## Parameters and Variables

In [4]:
window_T = 2.0       # window around the glitch
deltat = 0.2         # time btw triggers from the same cluster 
run = 'O3a'

# deltat -> tempottt..... window_T -> deltat ........ tempottt -> window_T

# parameters
cutSNR = True
SNRvalue_cut = 5.0
min_triggers = 10

clust = 'teste4_intervals'
suffix = "_C" + str(min_triggers) + "_dT" + str(deltat) + "_SNR" + str(SNRvalue_cut)

arq_name = 'SNRclustered' + suffix if cutSNR else 'clustered' + suffix

In [5]:
# folders
base = "/home/esanches/Projetos/tSNE/Virgo"
repository = "/home/esanches/Projetos/tSNE/Virgo/zrepository"

teste_path = "/home/esanches/Projetos/tSNE/Virgo/zrepository" + "/" + clust + "/" + run
vectors_path =  "/home/esanches/Projetos/tSNE/Virgo/zrepository" + "/" + clust + "/" + run + "/" + "tSNEandHDBSCAN"

# files
daily_clustering = "/home/esanches/Projetos/tSNE/Virgo/zrepository/" + clust + "/" + run + "/" + "dailyClustering" + suffix
clustered_path = "/home/esanches/Projetos/tSNE/Virgo/zrepository/" + clust + "/" + run + "/" + arq_name + ".csv"
non_matched_gdchar_name = vectors_path + "/non_matched_gdchar" + suffix + ".csv"

vecs_save = "/home/esanches/Projetos/tSNE/Virgo/zrepository/" + clust + "/" + run + "/" + "vecs" + suffix + ".npy"

In [6]:
daily_clustering

'/home/esanches/Projetos/tSNE/Virgo/zrepository/teste4_intervals/O3a/dailyClustering_C10_dT0.2_SNR5.0'

## Read the triggers

In [12]:
trigcols = ['time', 'frequency', 'snr']

dtypes = {
    'time': 'float64',
    'frequency': 'float32',
    'snr': 'float32'
}

path = base + "/" + "triggers_total_" + run + ".csv"

chunk_size = 10_000_000
count = 0
chunks = []

for chunk in pd.read_csv(path, usecols=trigcols, dtype=dtypes, chunksize=chunk_size):
    print(f'Reading chunk: {count}')
    chunks.append(chunk)
    count += 1

triggers = pd.concat(chunks, ignore_index=True)
triggers = triggers.sort_values('time').reset_index(drop=True)

unclustered = triggers.copy()

unclustered.tail()

Reading chunk: 0
Reading chunk: 1
Reading chunk: 2
Reading chunk: 3
Reading chunk: 4
Reading chunk: 5
Reading chunk: 6
Reading chunk: 7
Reading chunk: 8
Reading chunk: 9
Reading chunk: 10
Reading chunk: 11
Reading chunk: 12
Reading chunk: 13
Reading chunk: 14
Reading chunk: 15


,time,frequency,snr
156426272,1.253977e+09,22.160860,5.007397
156426273,1.253977e+09,49.533298,5.252314
156426274,1.253977e+09,74.707069,5.137199
156426275,1.253977e+09,60.831635,5.173639
156426276,1.253977e+09,49.533298,5.312661


## The routine

In [22]:
start_day_O3a = datetime(2019, 4, 1, 0, 0, 0)
end_day_O3a   = datetime(2019, 10, 1, 15, 0, 0)

start = start_day_O3a
end = end_day_O3a

In [23]:
intervals = pd.read_csv(repository + '/gwosc' + run + '.txt', sep=' ', header=None, names=['int1', 'int2', 'duration'])

In [24]:
df_glitches_final = routine(start, end, unclustered, intervals, min_triggers, deltat, output_dir=daily_clustering)

Day 2019-04-01 already processed. Skipping...
Day 2019-04-02 already processed. Skipping...
Day 2019-04-03 already processed. Skipping...
Day 2019-04-04 already processed. Skipping...
Day 2019-04-05 already processed. Skipping...
Day 2019-04-06 already processed. Skipping...
Day 2019-04-07 already processed. Skipping...
Day 2019-04-08 already processed. Skipping...
Day 2019-04-09 already processed. Skipping...
Day 2019-04-10 already processed. Skipping...
Day 2019-04-11 already processed. Skipping...
Day 2019-04-12 already processed. Skipping...
Day 2019-04-13 already processed. Skipping...
Day 2019-04-14 already processed. Skipping...
Day 2019-04-15 already processed. Skipping...
Day 2019-04-16 already processed. Skipping...
Day 2019-04-17 already processed. Skipping...
Day 2019-04-18 already processed. Skipping...
Day 2019-04-19 already processed. Skipping...
Day 2019-04-20 already processed. Skipping...
Day 2019-04-21 already processed. Skipping...
Day 2019-04-22 already processed. 

## Combine and save the .parquet files

These will put together the data obtained from Routine in only one .csv arquive

In [16]:
arq_name = "SNRclustered" + suffix if cutSNR else "clustered" + suffix - "_"
arquivo_csv_saida = repository + "/" + clust + "/" + run + "/" + arq_name + ".csv"

arquivos_parquet = sorted(glob.glob(os.path.join(daily_clustering, "glitches_*.parquet")))

df_list = [pd.read_parquet(f) for f in arquivos_parquet]
df_glitches_final = pd.concat(df_list, ignore_index=True)

df_glitches_final = df_glitches_final.drop(columns=['cluster']).sort_values('time').reset_index(drop=True)

if cutSNR:
    df_glitches_final = df_glitches_final[df_glitches_final['snr'] >= SNRvalue_cut].sort_values('time').reset_index(drop=True)

df_glitches_final.to_csv(arquivo_csv_saida, index=False)

In [17]:
df_glitches_final

,num_triggers,time,frequency,snr,tstart,tend,min_freq,max_freq
0,3,1.238166e+09,19.259594,5.440038,1.238166e+09,1.238166e+09,18.424749,19.259594
1,5,1.238166e+09,18.626778,5.224913,1.238166e+09,1.238166e+09,18.424749,19.494596
2,7,1.238166e+09,16.889482,6.232728,1.238166e+09,1.238166e+09,16.889482,17.731117
3,4,1.238166e+09,37.863487,5.503140,1.238166e+09,1.238166e+09,37.319275,37.863487
4,7,1.238166e+09,19.494596,5.412894,1.238166e+09,1.238166e+09,18.819546,19.816910
...,...,...,...,...,...,...,...,...
2766070,3,1.253977e+09,29.012165,5.223790,1.253977e+09,1.253977e+09,23.144554,29.012165
2766071,10,1.253977e+09,41.901775,5.938467,1.253977e+09,1.253977e+09,40.138248,42.512814
2766072,2,1.253977e+09,668.913757,5.074244,1.253977e+09,1.253977e+09,600.312500,668.913757
2766073,2,1.253977e+09,22.212175,5.067428,1.253977e+09,1.253977e+09,22.160860,22.212175


In [18]:
df_glitches_final['snr'].min()

np.float64(5.0)